# Coach DNA Profile Checks

This notebook validates the first-pass Coach DNA scoring outputs and pulls out the first round of interpretable findings.

## Goals
- confirm the exported scoring tables load correctly
- run basic QA on team counts, situation counts, and score ranges
- identify the strongest overall team signals
- inspect which situations drive the strongest differences
- review one team in detail against the league baseline
- generate findings that can later be used in the README or GitHub writeup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

# Robust project root detection
candidate_paths = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None

for path in candidate_paths:
    if (path / "python").exists() and (path / "outputs").exists():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root from notebook.")

OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROFILE_SEASON = 2025

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_TABLES_DIR:", OUTPUT_TABLES_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)

In [ ]:
ranked_team_summary = pd.read_csv(
    OUTPUT_TABLES_DIR / f"coach_dna_ranked_team_summary_{PROFILE_SEASON}.csv"
)

ranked_situation_scores = pd.read_csv(
    OUTPUT_TABLES_DIR / f"coach_dna_ranked_situation_scores_{PROFILE_SEASON}.csv"
)

top_signal_situations = pd.read_csv(
    OUTPUT_TABLES_DIR / f"coach_dna_top_signal_situations_by_team_{PROFILE_SEASON}.csv"
)

presentation_summary = pd.read_csv(
    OUTPUT_TABLES_DIR / f"coach_dna_team_summary_presentation_{PROFILE_SEASON}.csv"
)

team_baseline_features = pd.read_csv(
    PROCESSED_DATA_DIR / "team_baseline_features_2025_vs_2023_2025.csv"
)

print("ranked_team_summary:", ranked_team_summary.shape)
print("ranked_situation_scores:", ranked_situation_scores.shape)
print("top_signal_situations:", top_signal_situations.shape)
print("presentation_summary:", presentation_summary.shape)
print("team_baseline_features:", team_baseline_features.shape)

## 1. Basic QA Checks
These checks confirm the exported tables look structurally right before interpreting the results.

In [ ]:
qa_summary = {
    "team_summary_rows": len(ranked_team_summary),
    "unique_teams_in_team_summary": ranked_team_summary["team"].nunique(),
    "situation_score_rows": len(ranked_situation_scores),
    "unique_teams_in_situation_scores": ranked_situation_scores["team"].nunique(),
    "unique_situations_in_situation_scores": ranked_situation_scores["situation_name"].nunique(),
    "top_signal_rows": len(top_signal_situations),
    "unique_teams_in_top_signals": top_signal_situations["team"].nunique(),
}

pd.Series(qa_summary)

In [ ]:
ranked_situation_scores.groupby("team").size().sort_values().head(10)

In [ ]:
ranked_situation_scores.groupby("team").size().sort_values(ascending=False).head(10)

## 2. Team-Level Results
Start with the overall rankings to see which teams stand out most strongly in the first-pass scoring model.

In [ ]:
ranked_team_summary.head(10)

In [ ]:
ranked_team_summary.tail(10)

In [ ]:
ranked_team_summary["score_tier"].value_counts()

In [ ]:
ranked_team_summary[[
    "team",
    "overall_coach_dna_score",
    "tendency_signal_score_avg",
    "efficiency_signal_score_avg",
    "explosiveness_signal_score_avg",
    "stability_signal_score_avg",
    "top_signal_situation",
    "top_signal_score",
]].head(15)

## 3. Situation-Level Patterns
These checks show which situations tend to generate the strongest signals across the league.

In [ ]:
situation_strength = (
    ranked_situation_scores
    .groupby("situation_name", as_index=False)
    .agg(
        avg_adjusted_score=("coach_dna_score_adjusted", "mean"),
        max_adjusted_score=("coach_dna_score_adjusted", "max"),
        min_adjusted_score=("coach_dna_score_adjusted", "min"),
        avg_team_play_count=("team_play_count", "mean"),
        teams=("team", "nunique"),
    )
    .sort_values("avg_adjusted_score", ascending=False)
)

situation_strength

In [ ]:
situation_variance = (
    ranked_situation_scores
    .groupby("situation_name", as_index=False)
    .agg(
        score_std=("coach_dna_score_adjusted", "std"),
        score_range=("coach_dna_score_adjusted", lambda s: s.max() - s.min()),
        avg_sample=("team_play_count", "mean"),
    )
    .sort_values("score_std", ascending=False)
)

situation_variance

## 4. Top Signal Situations by Team
This helps identify the situations where each team looks most distinct relative to baseline.

In [ ]:
top_signal_situations.head(20)

In [ ]:
top_signal_situations["situation_name"].value_counts().head(15)

In [ ]:
top_signal_situations["tendency_profile_label"].value_counts()

In [ ]:
top_signal_situations["efficiency_profile_label"].value_counts()

## 5. Team Deep Dive
Change the `TEAM_CODE` below to inspect any team in more detail.

In [ ]:
TEAM_CODE = "BUF"

In [ ]:
team_summary_view = ranked_team_summary.loc[ranked_team_summary["team"] == TEAM_CODE]
team_summary_view

In [ ]:
team_situations = (
    ranked_situation_scores.loc[ranked_situation_scores["team"] == TEAM_CODE]
    .sort_values("situation_order")
)

team_situations[[
    "team",
    "situation_name",
    "team_play_count",
    "coach_dna_score_adjusted",
    "tendency_signal_score",
    "efficiency_signal_score",
    "explosiveness_signal_score",
    "stability_signal_score",
    "dropback_rate_delta",
    "rush_rate_delta",
    "shotgun_rate_delta",
    "no_huddle_rate_delta",
    "avg_epa_delta",
    "success_rate_delta",
    "explosive_play_rate_delta",
    "tendency_profile_label",
    "efficiency_profile_label",
]]

In [ ]:
team_situations.sort_values("coach_dna_score_adjusted", ascending=False).head(10)[[
    "situation_name",
    "team_play_count",
    "coach_dna_score_adjusted",
    "dropback_rate_delta",
    "rush_rate_delta",
    "avg_epa_delta",
    "success_rate_delta",
    "explosive_play_rate_delta",
    "tendency_profile_label",
    "efficiency_profile_label",
]]

In [ ]:
team_situations.sort_values("coach_dna_score_adjusted", ascending=True).head(10)[[
    "situation_name",
    "team_play_count",
    "coach_dna_score_adjusted",
    "dropback_rate_delta",
    "rush_rate_delta",
    "avg_epa_delta",
    "success_rate_delta",
    "explosive_play_rate_delta",
    "tendency_profile_label",
    "efficiency_profile_label",
]]

## 6. Sample Size Guardrails
These checks help keep us honest about situations that may be noisy because of smaller samples.

In [ ]:
ranked_situation_scores["team_sample_quality"].value_counts()

In [ ]:
ranked_situation_scores.loc[
    ranked_situation_scores["team_sample_quality"].isin(["thin", "very_thin"])
].sort_values(["team_sample_quality", "team_play_count", "team"]).head(30)

In [ ]:
small_sample_summary = (
    ranked_situation_scores
    .groupby(["situation_name", "team_sample_quality"], as_index=False)
    .size()
    .sort_values(["situation_name", "team_sample_quality"])
)

small_sample_summary.head(50)

## 7. Quick Finding Builder
These tables help turn the output into portfolio-ready statements.

In [ ]:
top_overall = ranked_team_summary.head(5)[["rank", "team", "overall_coach_dna_score", "top_signal_situation", "top_signal_score"]]
top_overall

In [ ]:
best_situations = (
    ranked_situation_scores
    .sort_values("coach_dna_score_adjusted", ascending=False)
    .head(15)[[
        "rank",
        "team",
        "situation_name",
        "team_play_count",
        "coach_dna_score_adjusted",
        "tendency_profile_label",
        "efficiency_profile_label",
    ]]
)

best_situations

In [ ]:
most_run_heavy = (
    ranked_situation_scores
    .sort_values("rush_rate_delta", ascending=False)
    .head(15)[[
        "team",
        "situation_name",
        "team_play_count",
        "rush_rate_delta",
        "avg_epa_delta",
        "success_rate_delta",
    ]]
)

most_run_heavy

In [ ]:
most_dropback_heavy = (
    ranked_situation_scores
    .sort_values("dropback_rate_delta", ascending=False)
    .head(15)[[
        "team",
        "situation_name",
        "team_play_count",
        "dropback_rate_delta",
        "avg_epa_delta",
        "success_rate_delta",
    ]]
)

most_dropback_heavy

In [ ]:
most_efficient = (
    ranked_situation_scores
    .sort_values("avg_epa_delta", ascending=False)
    .head(15)[[
        "team",
        "situation_name",
        "team_play_count",
        "avg_epa_delta",
        "success_rate_delta",
        "explosive_play_rate_delta",
    ]]
)

most_efficient

## 8. Notes
Use this section to write down 5–10 findings that are worth carrying into the README, GitHub post, or interview narrative.

### Draft findings
- 
- 
- 
- 
- 